In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, losses, metrics
import numpy as np

print("TensorFlow:", tf.__version__)
print("Keras:     ", keras.__version__)
print("GPUs:      ", tf.config.list_physical_devices('GPU'))
print("Eager mode:", tf.executing_eagerly())

# TensorFlow 2 API — Comprehensive Reference
## Understanding TensorFlow Under the Hood with Python API Examples

This notebook provides a comprehensive guide to TensorFlow 2 functions with practical Python examples.
Sections mirror the structure of the PyTorch API guide so the two can be compared side by side.

**Contents:**
1. Tensor Creation Functions
2. Tensor Properties & Attributes
3. Tensor Reshaping & Manipulation
4. Mathematical Operations
5. Linear Algebra Operations
6. Reduction Operations
7. Activation Functions
8. Loss Functions
9. Neural Network Layers (Keras)
10. Optimizers & Learning Rate Schedulers
11. Automatic Differentiation — GradientTape
12. Device Management
13. Comparison & Logical Operations
14. Data Pipelines — tf.data
15. Serialization & Model Saving
16. Utilities & Helpers
17. Complete Training Example
18. Summary & Quick Reference

# 1. Tensor Creation Functions

## Basic Tensor Creation

In [ ]:
# tf.constant() — immutable tensor from data
scalar = tf.constant(3.14)
vector = tf.constant([1.0, 2.0, 3.0, 4.0, 5.0])
matrix = tf.constant([[1, 2, 3], [4, 5, 6]], dtype=tf.float32)

print("scalar:", scalar)
print("vector:", vector)
print("matrix:\n", matrix.numpy())

# tf.Variable — mutable tensor (used for trainable parameters)
var = tf.Variable([1.0, 2.0, 3.0])
var.assign_add([10.0, 10.0, 10.0])
print("variable after assign_add:", var.numpy())

In [ ]:
# Factory functions
zeros   = tf.zeros([3, 4])                    # 3×4 all-zeros
ones    = tf.ones([3, 4])                     # 3×4 all-ones
filled  = tf.fill([2, 3], 7.0)               # 2×3 filled with 7
eye_m   = tf.eye(4)                           # 4×4 identity
ranged  = tf.range(0, 10, delta=2)            # [0, 2, 4, 6, 8]
linsp   = tf.linspace(0.0, 1.0, 6)           # 6 evenly spaced values

print("zeros shape:",  zeros.shape)
print("eye:\n",        eye_m.numpy())
print("range:",        ranged.numpy())
print("linspace:",     linsp.numpy())

# _like variants — match shape of an existing tensor
template     = tf.random.normal([2, 3])
zeros_like   = tf.zeros_like(template)
ones_like    = tf.ones_like(template)
print("zeros_like shape:", zeros_like.shape)

In [ ]:
# Random tensor creation
tf.random.set_seed(42)

# tf.random.normal — standard normal N(0, 1)
norm   = tf.random.normal([2, 3])

# tf.random.normal with mean/stddev
norm2  = tf.random.normal([2, 3], mean=5.0, stddev=2.0)

# tf.random.uniform — uniform [minval, maxval)
uni    = tf.random.uniform([2, 3])               # [0, 1)
uni2   = tf.random.uniform([2, 3], 0, 10, dtype=tf.int32)

# tf.random.shuffle — shuffle along first axis
shuf   = tf.random.shuffle(tf.constant([1, 2, 3, 4, 5]))

print("normal:\n",         norm.numpy())
print("uniform int:\n",    uni2.numpy())
print("shuffled:",         shuf.numpy())

# 2. Tensor Properties & Attributes

In [ ]:
t = tf.random.normal([3, 4, 5])

# Shape and size
print("t.shape:       ", t.shape)            # TensorShape([3, 4, 5])
print("t.shape[0]:    ", t.shape[0])         # 3
print("tf.rank(t):    ", tf.rank(t).numpy()) # 3  (number of dimensions)
print("t.ndim:        ", t.ndim)             # 3
print("tf.size(t):    ", tf.size(t).numpy()) # 60 (total elements)

# Data type
print("t.dtype:       ", t.dtype)            # tf.float32
print("dtype.is_float:", t.dtype.is_floating)

# Device
print("t.device:      ", t.device)

# Convert to NumPy
arr = t.numpy()                              # ndarray shape (3, 4, 5)
print("numpy shape:   ", arr.shape)

# 3. Tensor Reshaping & Manipulation

In [ ]:
t = tf.cast(tf.range(24), tf.float32)   # shape (24,)

# reshape — total elements must match
a = tf.reshape(t, [2, 3, 4])   # (2, 3, 4)
b = tf.reshape(t, [6, 4])      # (6, 4)
c = tf.reshape(t, [-1, 6])     # (-1 inferred) → (4, 6)

print("reshape [2,3,4]:", a.shape)
print("reshape [6,4]:  ", b.shape)
print("reshape [-1,6]: ", c.shape)

In [ ]:
m = tf.random.normal([3, 4])

# expand_dims / squeeze
u0  = tf.expand_dims(m, 0)        # (1, 3, 4)
u1  = tf.expand_dims(m, 1)        # (3, 1, 4)
u2  = tf.expand_dims(m, -1)       # (3, 4, 1)
sq  = tf.squeeze(u0, 0)           # (3, 4)
sq2 = tf.squeeze(u0)              # removes all size-1 dims → (3, 4)

print("expand axis=0: ", u0.shape)
print("expand axis=1: ", u1.shape)
print("squeeze:       ", sq.shape)

# transpose / permute
t2 = tf.transpose(m)                       # (4, 3)
a  = tf.random.normal([2, 3, 4])
p  = tf.transpose(a, perm=[2, 0, 1])      # (4, 2, 3)
print("transpose:     ", t2.shape)
print("permute:       ", p.shape)

# concat and stack
x = tf.ones([2, 3])
y = tf.zeros([2, 3])
cat_r = tf.concat([x, y], axis=0)    # (4, 3)
cat_c = tf.concat([x, y], axis=1)    # (2, 6)
stk   = tf.stack([x, y], axis=0)     # (2, 2, 3) — new dim
print("concat rows:   ", cat_r.shape)
print("stack:         ", stk.shape)

# 4. Mathematical Operations

In [ ]:
a = tf.constant([1.0, 2.0, 3.0, 4.0])
b = tf.constant([10.0, 20.0, 30.0, 40.0])

# Element-wise arithmetic (operator overloads and functional forms)
print("a + b:    ", (a + b).numpy())            # tf.add
print("b - a:    ", (b - a).numpy())            # tf.subtract
print("a * b:    ", (a * b).numpy())            # tf.multiply
print("b / a:    ", (b / a).numpy())            # tf.divide
print("a ** 2:   ", (a ** 2).numpy())           # tf.pow
print("a % 3:    ", tf.math.mod(a, 3.0).numpy())

# Scalar broadcasting
print("a * 3:    ", (a * 3.0).numpy())
print("a + 100:  ", (a + 100.0).numpy())

# Element-wise math functions
t = tf.constant([-2.0, -1.0, 0.0, 1.0, 2.0])
print("abs:      ", tf.abs(t).numpy())
print("exp:      ", tf.exp(t).numpy())
print("log(|t|): ", tf.math.log(tf.abs(t) + 1e-6).numpy())
print("sqrt(|t|):", tf.sqrt(tf.abs(t)).numpy())
print("sin:      ", tf.sin(t).numpy())
print("clip:     ", tf.clip_by_value(t, -1.0, 1.0).numpy())

# 5. Linear Algebra Operations

In [ ]:
A = tf.random.normal([3, 4], seed=0)
B = tf.random.normal([4, 5], seed=1)

# Matrix multiply
C  = A @ B                                # (3, 5) — operator overload
C2 = tf.linalg.matmul(A, B)             # same
print("A @ B shape:", C.shape)

# Dot product (1-D)
u  = tf.constant([1.0, 2.0, 3.0])
v  = tf.constant([4.0, 5.0, 6.0])
dot = tf.tensordot(u, v, axes=1)
print("dot product:", dot.numpy())        # 32

# Outer product
outer = tf.tensordot(u, v, axes=0)
print("outer shape:", outer.shape)        # (3, 3)

# SVD — s: singular values, u: left vectors, v: right vectors
s, u_tf, v_tf = tf.linalg.svd(A)
print("SVD s:", s.numpy())

# QR decomposition
q, r = tf.linalg.qr(A)
print("QR q shape:", q.shape, "r shape:", r.shape)

# Inverse, determinant, trace
sq_A = A @ tf.transpose(A)               # 3×3 positive definite
Ainv = tf.linalg.inv(sq_A)
det  = tf.linalg.det(sq_A)
tr   = tf.linalg.trace(sq_A)
print("det: ", det.numpy())
print("trace:", tr.numpy())

# Norms
frob    = tf.norm(A)                     # Frobenius
col_l2  = tf.norm(A, axis=0)            # per-column L2
print("Frobenius norm:", frob.numpy())

# 6. Reduction Operations

In [ ]:
m = tf.constant([[1.0, 2.0, 3.0],
                 [4.0, 5.0, 6.0]])

# Global reductions
print("sum:        ", tf.reduce_sum(m).numpy())
print("mean:       ", tf.reduce_mean(m).numpy())
print("max:        ", tf.reduce_max(m).numpy())
print("min:        ", tf.reduce_min(m).numpy())
print("prod:       ", tf.reduce_prod(m).numpy())
print("std:        ", tf.math.reduce_std(m).numpy())
print("variance:   ", tf.math.reduce_variance(m).numpy())

# Along an axis
print("col sum:    ", tf.reduce_sum(m, axis=0).numpy())           # [5, 7, 9]
print("row sum:    ", tf.reduce_sum(m, axis=1).numpy())           # [6, 15]
print("row mean k: ", tf.reduce_mean(m, axis=1, keepdims=True).numpy())  # [[2],[5]]

# Argmax / argmin
print("argmax col: ", tf.argmax(m, axis=0).numpy())               # [1, 1, 1]
print("argmax row: ", tf.argmax(m, axis=1).numpy())               # [2, 2]

# Sort
flat     = tf.reshape(m, [-1])
sorted_v = tf.sort(flat)
sort_idx = tf.argsort(flat)
cumsum   = tf.cumsum(flat)
print("sorted:     ", sorted_v.numpy())
print("cumsum:     ", cumsum.numpy())

# Unique with counts
vals, idx, cnts = tf.unique_with_counts(tf.constant([1, 2, 2, 3, 3, 3]))
print("unique:     ", vals.numpy())
print("counts:     ", cnts.numpy())

# 7. Activation Functions

In [ ]:
import tensorflow as tf

x = tf.constant([-3.0, -1.0, 0.0, 1.0, 3.0])

# Functional forms (tf.nn)
print("relu:        ", tf.nn.relu(x).numpy())
print("leaky_relu:  ", tf.nn.leaky_relu(x, alpha=0.1).numpy())
print("elu:         ", tf.nn.elu(x).numpy())
print("selu:        ", tf.nn.selu(x).numpy())
print("gelu:        ", tf.nn.gelu(x).numpy())
print("sigmoid:     ", tf.sigmoid(x).numpy())
print("tanh:        ", tf.tanh(x).numpy())
print("softplus:    ", tf.nn.softplus(x).numpy())
print("swish:       ", tf.nn.swish(x).numpy())

# Softmax (needs 2-D or explicit axis)
logits = tf.constant([[1.0, 2.0, 3.0],
                       [0.5, 1.5, 0.5]])
probs  = tf.nn.softmax(logits, axis=-1)
print("softmax:\n",      probs.numpy())

# log_softmax
log_p  = tf.nn.log_softmax(logits, axis=-1)
print("log_softmax:\n", log_p.numpy())

# As Keras layers (track state / support serialization)
relu_layer    = tf.keras.layers.ReLU()
softmax_layer = tf.keras.layers.Softmax()
gelu_layer    = tf.keras.layers.Activation('gelu')

# 8. Loss Functions

In [ ]:
from tensorflow.keras import losses

# ── Regression losses ─────────────────────────────────────────────────────────
y_true_reg = tf.constant([1.0, 2.0, 3.0, 4.0])
y_pred_reg = tf.constant([1.1, 1.9, 3.2, 3.8])

mse  = losses.MeanSquaredError()(y_true_reg, y_pred_reg)
mae  = losses.MeanAbsoluteError()(y_true_reg, y_pred_reg)
huber = losses.Huber(delta=1.0)(y_true_reg, y_pred_reg)

print(f"MSE:   {mse.numpy():.4f}")
print(f"MAE:   {mae.numpy():.4f}")
print(f"Huber: {huber.numpy():.4f}")

# ── Classification losses ──────────────────────────────────────────────────────
y_true_cls = tf.constant([0, 1, 2])          # integer labels
logits_cls = tf.constant([
    [2.0, 0.5, 0.1],
    [0.2, 3.0, 0.5],
    [0.1, 0.3, 2.8]
])

# SparseCategoricalCrossentropy (integer labels, logits)
sce  = losses.SparseCategoricalCrossentropy(from_logits=True)(y_true_cls, logits_cls)
print(f"SparseCE: {sce.numpy():.4f}")

# CategoricalCrossentropy (one-hot labels, probabilities)
y_onehot = tf.one_hot(y_true_cls, depth=3)
probs    = tf.nn.softmax(logits_cls)
ce       = losses.CategoricalCrossentropy()(y_onehot, probs)
print(f"CE:       {ce.numpy():.4f}")

# BinaryCrossentropy
y_bin   = tf.constant([1.0, 0.0, 1.0, 1.0])
p_bin   = tf.constant([0.9, 0.1, 0.8, 0.4])
bce     = losses.BinaryCrossentropy()(y_bin, p_bin)
print(f"BCE:      {bce.numpy():.4f}")

# 9. Neural Network Layers (Keras)

In [ ]:
from tensorflow.keras import layers

# ── Linear ────────────────────────────────────────────────────────────────────
dense   = layers.Dense(64, activation='relu')
x       = tf.random.normal([4, 16])
out     = dense(x)
print("Dense out:", out.shape)    # (4, 64)

# ── Convolution ───────────────────────────────────────────────────────────────
conv2d  = layers.Conv2D(32, kernel_size=3, padding='same', activation='relu')
img     = tf.random.normal([8, 28, 28, 1])   # NHWC
feat    = conv2d(img)
print("Conv2D out:", feat.shape)  # (8, 28, 28, 32)

# ── Normalisation ─────────────────────────────────────────────────────────────
bn      = layers.BatchNormalization()
ln      = layers.LayerNormalization()
bn_out  = bn(out, training=True)
print("BatchNorm out:", bn_out.shape)

# ── Recurrent ─────────────────────────────────────────────────────────────────
lstm    = layers.LSTM(32, return_sequences=True)
seq     = tf.random.normal([4, 10, 8])   # (batch, time, features)
lstm_out = lstm(seq)
print("LSTM out:", lstm_out.shape)       # (4, 10, 32)

# ── Regularisation ────────────────────────────────────────────────────────────
drop    = layers.Dropout(0.3)
d_out   = drop(out, training=True)

# ── Embedding ─────────────────────────────────────────────────────────────────
emb     = layers.Embedding(vocab_size=1000, output_dim=64)
tokens  = tf.constant([[1, 5, 32], [9, 0, 4]])
emb_out = emb(tokens)
print("Embedding out:", emb_out.shape)   # (2, 3, 64)

# ── Pooling ───────────────────────────────────────────────────────────────────
pool2d  = layers.MaxPooling2D(pool_size=2)
p_out   = pool2d(feat)
print("MaxPool2D out:", p_out.shape)     # (8, 14, 14, 32)

gap     = layers.GlobalAveragePooling2D()
gap_out = gap(feat)
print("GAP out:", gap_out.shape)         # (8, 32)

In [ ]:
# ── Building models ───────────────────────────────────────────────────────────

# Sequential API
seq_model = tf.keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(16,)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dense(3)
])
seq_model.summary()

# Functional API
inputs  = tf.keras.Input(shape=(16,))
x       = layers.Dense(128, activation='relu')(inputs)
x       = layers.Dropout(0.3)(x)
outputs = layers.Dense(3)(x)
func_model = tf.keras.Model(inputs, outputs, name='functional_mlp')

# Subclassing
class MLP(tf.keras.Model):
    def __init__(self, hidden, num_classes):
        super().__init__()
        self.fc1  = layers.Dense(hidden, activation='relu')
        self.drop = layers.Dropout(0.3)
        self.fc2  = layers.Dense(num_classes)

    def call(self, x, training=False):
        x = self.fc1(x)
        x = self.drop(x, training=training)
        return self.fc2(x)

mlp = MLP(hidden=64, num_classes=3)
print("Parameters:", mlp.count_params())

# 10. Optimizers & Learning Rate Schedulers

In [ ]:
from tensorflow.keras import optimizers

# ── Common optimizers ─────────────────────────────────────────────────────────
sgd   = optimizers.SGD(learning_rate=0.01, momentum=0.9, nesterov=True)
adam  = optimizers.Adam(learning_rate=1e-3, beta_1=0.9, beta_2=0.999)
rms   = optimizers.RMSprop(learning_rate=1e-3, rho=0.9)
adagrad = optimizers.Adagrad(learning_rate=1e-2)

print("Adam config:", adam.get_config())

# ── Learning rate schedules ───────────────────────────────────────────────────
# Exponential decay
exp_sched = optimizers.schedules.ExponentialDecay(
    initial_learning_rate=1e-3,
    decay_steps=1000,
    decay_rate=0.96,
    staircase=True
)

# Cosine decay
cos_sched = optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-3,
    decay_steps=5000
)

# Polynomial decay
poly_sched = optimizers.schedules.PolynomialDecay(
    initial_learning_rate=1e-2,
    decay_steps=10000,
    end_learning_rate=1e-5
)

opt_with_sched = optimizers.Adam(learning_rate=cos_sched)

# ── Manual gradient application ───────────────────────────────────────────────
model = MLP(64, 3)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = optimizers.Adam(1e-3)

x_batch = tf.random.normal([32, 16])
y_batch = tf.cast(tf.random.uniform([32], maxval=3, dtype=tf.int32), tf.int32)

with tf.GradientTape() as tape:
    logits = model(x_batch, training=True)
    loss   = loss_fn(y_batch, logits)

grads = tape.gradient(loss, model.trainable_variables)
optimizer.apply_gradients(zip(grads, model.trainable_variables))

print(f"Step loss: {loss.numpy():.4f}")

# 11. Automatic Differentiation — GradientTape

In [ ]:
# Basic gradient computation
x = tf.Variable(2.0)
w = tf.Variable(3.0)
b = tf.Variable(1.0)

with tf.GradientTape() as tape:
    y = w * x + b    # 3*2 + 1 = 7

grads = tape.gradient(y, [w, x, b])
print("dy/dw:", grads[0].numpy())   # 2  (= x)
print("dy/dx:", grads[1].numpy())   # 3  (= w)
print("dy/db:", grads[2].numpy())   # 1

# Vector function
a = tf.Variable([1.0, 2.0, 3.0])
with tf.GradientTape() as tape:
    out = tf.reduce_sum(a ** 2)
grad_a = tape.gradient(out, a)
print("d(sum a²)/da:", grad_a.numpy())  # [2, 4, 6]

# Watch a non-Variable tensor
const = tf.constant(4.0)
with tf.GradientTape() as tape:
    tape.watch(const)
    f = const ** 2
print("d(x²)/dx at 4:", tape.gradient(f, const).numpy())  # 8

# Persistent tape — multiple gradient calls
x2 = tf.Variable(3.0)
with tf.GradientTape(persistent=True) as tape:
    y2 = x2 ** 2
    z  = x2 ** 3
dy = tape.gradient(y2, x2)   # 6
dz = tape.gradient(z,  x2)   # 27
del tape
print("dy2/dx:", dy.numpy(), "  dz/dx:", dz.numpy())

# Second-order gradient
x3 = tf.Variable(3.0)
with tf.GradientTape() as outer:
    with tf.GradientTape() as inner:
        y3 = x3 ** 3
    dy_dx = inner.gradient(y3, x3)
d2y = outer.gradient(dy_dx, x3)
print("d²(x³)/dx² at 3:", d2y.numpy())  # 18  (= 6*x)

# 12. Device Management

In [ ]:
# List all physical devices
print("All devices:")
for d in tf.config.list_physical_devices():
    print(" ", d)

# Check GPU availability
gpus = tf.config.list_physical_devices('GPU')
print("\nGPUs available:", len(gpus))

# Place operations on CPU
with tf.device('/CPU:0'):
    t_cpu = tf.random.normal([3, 3])
print("CPU tensor device:", t_cpu.device)

# Place operations on GPU (only runs if GPU is available)
if gpus:
    with tf.device('/GPU:0'):
        t_gpu = tf.random.normal([3, 3])
    print("GPU tensor device:", t_gpu.device)
else:
    print("No GPU — operations fall back to CPU automatically")

# Enable memory growth (best practice)
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

# Multi-GPU training with MirroredStrategy
strategy = tf.distribute.MirroredStrategy()
print("\nReplicas:", strategy.num_replicas_in_sync)

# 13. Comparison & Logical Operations

In [ ]:
x = tf.constant([1.0, 5.0, 3.0, 7.0, 2.0])
y = tf.constant([2.0, 5.0, 1.0, 6.0, 2.0])

# Element-wise comparison
print("x > 3:  ", tf.greater(x, 3.0).numpy())
print("x >= 3: ", tf.greater_equal(x, 3.0).numpy())
print("x < 3:  ", tf.less(x, 3.0).numpy())
print("x == y: ", tf.equal(x, y).numpy())
print("x != y: ", tf.not_equal(x, y).numpy())

# Operator overloads work too
print("x > y:  ", (x > y).numpy())

# Logical ops (on bool tensors)
a = x > 2.0
b = x < 6.0
print("a & b:  ", tf.logical_and(a, b).numpy())
print("a | b:  ", tf.logical_or(a, b).numpy())
print("~a:     ", tf.logical_not(a).numpy())

# tf.where — conditional select
result = tf.where(x > 3.0, x, tf.zeros_like(x))
print("where:  ", result.numpy())     # [0, 5, 0, 7, 0]

# Boolean mask
mask     = x > 3.0
selected = tf.boolean_mask(x, mask)
print("masked: ", selected.numpy())   # [5, 7]

# reduce_any / reduce_all
print("any > 6:", tf.reduce_any(x > 6.0).numpy())
print("all > 0:", tf.reduce_all(x > 0.0).numpy())

# 14. Data Pipelines — tf.data

In [ ]:
import numpy as np

# Build a sample dataset
X = np.random.randn(1000, 16).astype(np.float32)
y = (X.sum(axis=1) > 0).astype(np.int32) % 3

# ── From NumPy arrays ─────────────────────────────────────────────────────────
dataset = tf.data.Dataset.from_tensor_slices((X, y))
print("Dataset element spec:", dataset.element_spec)

# Standard shuffle → batch → prefetch pipeline
train_ds = (dataset
    .shuffle(buffer_size=1000, seed=42)
    .batch(64)
    .prefetch(tf.data.AUTOTUNE))

# Peek at one batch
for x_b, y_b in train_ds.take(1):
    print("Batch x:", x_b.shape, "  y:", y_b.shape)

# ── Transformations ───────────────────────────────────────────────────────────

# map — apply a function to every element
normalized_ds = dataset.map(
    lambda x, y: (x / tf.norm(x, keepdims=True), y),
    num_parallel_calls=tf.data.AUTOTUNE
)

# filter — keep elements satisfying a predicate
class0_ds = dataset.filter(lambda x, y: tf.equal(y, 0))

# cache — keep dataset in memory after first epoch
cached_ds = dataset.cache()

# repeat — loop over the dataset multiple times
repeated_ds = dataset.repeat(3)

# zip — combine two datasets
ds_a = tf.data.Dataset.from_tensor_slices(X)
ds_b = tf.data.Dataset.from_tensor_slices(y)
zipped = tf.data.Dataset.zip((ds_a, ds_b))

# ── From a generator ──────────────────────────────────────────────────────────
def data_gen():
    for i in range(100):
        yield np.random.randn(16).astype(np.float32), np.int32(i % 3)

gen_ds = tf.data.Dataset.from_generator(
    data_gen,
    output_signature=(
        tf.TensorSpec(shape=(16,), dtype=tf.float32),
        tf.TensorSpec(shape=(),    dtype=tf.int32)
    )
)
print("Generator dataset spec:", gen_ds.element_spec)

# 15. Serialization & Model Saving

In [ ]:
import os, tempfile

model = MLP(64, 3)
# Build the model by passing a dummy input
_ = model(tf.zeros([1, 16]))

tmp = tempfile.mkdtemp()

# ── Keras native format (.keras) ──────────────────────────────────────────────
keras_path = os.path.join(tmp, 'model.keras')
model.save(keras_path)
loaded_keras = tf.keras.models.load_model(keras_path)
print("Keras reload OK:", loaded_keras(tf.zeros([1, 16])).shape)

# ── SavedModel format (for TF Serving) ────────────────────────────────────────
sm_path = os.path.join(tmp, 'saved_model')
model.save(sm_path)
loaded_sm = tf.keras.models.load_model(sm_path)
print("SavedModel reload OK:", loaded_sm(tf.zeros([1, 16])).shape)

# ── Weights only ──────────────────────────────────────────────────────────────
h5_path = os.path.join(tmp, 'weights.h5')
model.save_weights(h5_path)
model.load_weights(h5_path)
print("Weights reload OK")

# ── TF Checkpoint (low-level) ─────────────────────────────────────────────────
opt   = tf.keras.optimizers.Adam(1e-3)
ckpt  = tf.train.Checkpoint(model=model, optimizer=opt)
mgr   = tf.train.CheckpointManager(ckpt, os.path.join(tmp, 'ckpts'), max_to_keep=3)
mgr.save()

ckpt.restore(mgr.latest_checkpoint)
print("Checkpoint restored from:", mgr.latest_checkpoint)

# ── Export as NumPy weights dict ──────────────────────────────────────────────
weights_dict = {v.name: v.numpy() for v in model.trainable_variables}
npy_path     = os.path.join(tmp, 'weights.npy')
np.save(npy_path, weights_dict)
restored     = np.load(npy_path, allow_pickle=True).item()
print("NumPy weight keys:", list(restored.keys())[:3])

# 16. Utilities & Helpers

In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────────
tf.random.set_seed(42)

# ── One-hot encoding ──────────────────────────────────────────────────────────
labels  = tf.constant([0, 1, 2, 1])
one_hot = tf.one_hot(labels, depth=3)
print("one_hot:\n", one_hot.numpy())

# ── Padding and masking ───────────────────────────────────────────────────────
seqs = tf.constant([[1, 2, 3], [4, 5, 0]])
padded = tf.pad(seqs, [[0, 0], [0, 2]])   # pad 2 cols on right
print("padded:\n", padded.numpy())

# ── tf.function — compile for speed ──────────────────────────────────────────
@tf.function
def train_step(model, x, y, opt, loss_fn):
    with tf.GradientTape() as tape:
        logits = model(x, training=True)
        loss   = loss_fn(y, logits)
    grads = tape.gradient(loss, model.trainable_variables)
    opt.apply_gradients(zip(grads, model.trainable_variables))
    return loss

# ── Ragged tensors (variable-length sequences) ────────────────────────────────
ragged = tf.ragged.constant([[1, 2, 3], [4, 5], [6]])
print("ragged:", ragged)
print("ragged[0]:", ragged[0].numpy())

# ── SparseTensor ──────────────────────────────────────────────────────────────
sparse = tf.SparseTensor(
    indices=[[0, 1], [1, 2]],
    values=[1.0, 2.0],
    dense_shape=[3, 4]
)
dense_from_sparse = tf.sparse.to_dense(sparse)
print("dense from sparse:\n", dense_from_sparse.numpy())

# ── Gradient clipping ─────────────────────────────────────────────────────────
grad_list  = [tf.random.normal([10, 10]) for _ in range(3)]
clipped, global_norm = tf.clip_by_global_norm(grad_list, clip_norm=1.0)
print("global norm before clip:", global_norm.numpy())

# ── Mixed precision ───────────────────────────────────────────────────────────
# tf.keras.mixed_precision.set_global_policy('mixed_float16')  # uncomment on GPU

# ── Profiler ─────────────────────────────────────────────────────────────────
# tf.profiler.experimental.start('logdir')  # start profiling
# ... run code ...
# tf.profiler.experimental.stop()

# 17. Complete Training Example

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

tf.random.set_seed(42)
np.random.seed(42)

# ── Dataset ───────────────────────────────────────────────────────────────────
def make_data(n, features=16, classes=3):
    X = np.random.randn(n, features).astype(np.float32)
    y = (X.sum(axis=1) > 0).astype(np.int32) % classes
    return X, y

X_train, y_train = make_data(800)
X_val,   y_val   = make_data(200)

BATCH = 64
train_ds = (tf.data.Dataset
    .from_tensor_slices((X_train, y_train))
    .shuffle(800)
    .batch(BATCH)
    .prefetch(tf.data.AUTOTUNE))

val_ds = (tf.data.Dataset
    .from_tensor_slices((X_val, y_val))
    .batch(BATCH)
    .prefetch(tf.data.AUTOTUNE))

# ── Model ─────────────────────────────────────────────────────────────────────
class Classifier(keras.Model):
    def __init__(self, hidden, num_classes):
        super().__init__()
        self.fc1  = layers.Dense(hidden, activation='relu')
        self.bn1  = layers.BatchNormalization()
        self.drop = layers.Dropout(0.3)
        self.fc2  = layers.Dense(hidden // 2, activation='relu')
        self.bn2  = layers.BatchNormalization()
        self.out  = layers.Dense(num_classes)

    def call(self, x, training=False):
        x = self.bn1(self.fc1(x), training=training)
        x = self.drop(x, training=training)
        x = self.bn2(self.fc2(x), training=training)
        return self.out(x)

model   = Classifier(hidden=128, num_classes=3)
opt     = keras.optimizers.Adam(1e-3)
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
acc_fn  = keras.metrics.SparseCategoricalAccuracy()

# ── Training loop ─────────────────────────────────────────────────────────────
@tf.function
def train_step(x, y):
    with tf.GradientTape() as tape:
        logits = model(x, training=True)
        loss   = loss_fn(y, logits)
    grads = tape.gradient(loss, model.trainable_variables)
    opt.apply_gradients(zip(grads, model.trainable_variables))
    acc_fn.update_state(y, logits)
    return loss

@tf.function
def val_step(x, y):
    logits = model(x, training=False)
    loss   = loss_fn(y, logits)
    acc_fn.update_state(y, logits)
    return loss

EPOCHS = 50
for epoch in range(1, EPOCHS + 1):
    acc_fn.reset_state()
    epoch_loss = 0.0
    steps = 0

    for x_b, y_b in train_ds:
        epoch_loss += train_step(x_b, y_b).numpy()
        steps      += 1

    if epoch % 10 == 0:
        train_acc = acc_fn.result().numpy()

        acc_fn.reset_state()
        val_loss = 0.0
        v_steps  = 0
        for x_b, y_b in val_ds:
            val_loss += val_step(x_b, y_b).numpy()
            v_steps  += 1

        print(f"Epoch {epoch:3d}  "
              f"train_loss={epoch_loss/steps:.4f}  train_acc={train_acc:.4f}  "
              f"val_loss={val_loss/v_steps:.4f}  val_acc={acc_fn.result().numpy():.4f}")

# ── Final evaluation via compile/evaluate ────────────────────────────────────
model.compile(optimizer=opt,
              loss=loss_fn,
              metrics=[keras.metrics.SparseCategoricalAccuracy(name='acc')])
val_loss, val_acc = model.evaluate(val_ds, verbose=0)
print(f"\nFinal val accuracy: {val_acc*100:.1f}%")

# ── Save ─────────────────────────────────────────────────────────────────────
import tempfile, os
tmp = tempfile.mkdtemp()
model.save(os.path.join(tmp, 'classifier.keras'))
print("Model saved")

reloaded = keras.models.load_model(os.path.join(tmp, 'classifier.keras'))
_, reload_acc = reloaded.evaluate(val_ds, verbose=0)
print(f"Reloaded accuracy: {reload_acc*100:.1f}%")

# 18. Summary & Quick Reference

## TensorFlow 2 API Overview

This notebook covered all major TensorFlow / Keras functions organised by category:

### Core Tensor Operations
- **Creation**: `tf.constant()`, `tf.Variable()`, `tf.zeros()`, `tf.ones()`, `tf.fill()`, `tf.eye()`, `tf.range()`, `tf.linspace()`, `tf.random.normal()`, `tf.random.uniform()`
- **Shape**: `tf.reshape()`, `tf.expand_dims()`, `tf.squeeze()`, `tf.transpose()`, `tf.concat()`, `tf.stack()`, `tf.split()`
- **Indexing**: standard `[]` slicing, `tf.gather()`, `tf.gather_nd()`, `tf.boolean_mask()`, `tf.tensor_scatter_nd_update()`

### Math & Linear Algebra
- **Element-wise**: `+`, `-`, `*`, `/`, `**`, `tf.abs()`, `tf.exp()`, `tf.sqrt()`, `tf.clip_by_value()`
- **Matrix**: `@` / `tf.linalg.matmul()`, `tf.tensordot()`, `tf.linalg.svd()`, `tf.linalg.inv()`, `tf.norm()`
- **Reduction**: `tf.reduce_sum()`, `tf.reduce_mean()`, `tf.reduce_max()`, `tf.argmax()`, `tf.sort()`

### Autograd
- **Gradient**: `tf.GradientTape()` → `tape.gradient(y, variables)`
- **Persistent**: `tf.GradientTape(persistent=True)` for multiple calls
- **Non-Variable**: `tape.watch(tensor)`

### Neural Networks (Keras)
- **Layers**: `Dense`, `Conv2D`, `LSTM`, `BatchNormalization`, `Dropout`, `Embedding`, `MaxPooling2D`
- **Models**: `Sequential`, `Functional API (keras.Model)`, `Subclassing`
- **Activations**: `relu`, `sigmoid`, `tanh`, `softmax`, `gelu`, `swish`
- **Losses**: `SparseCategoricalCrossentropy`, `MeanSquaredError`, `BinaryCrossentropy`
- **Optimizers**: `Adam`, `SGD`, `RMSprop` with LR schedules

### Data & I/O
- **Pipelines**: `tf.data.Dataset.from_tensor_slices()`, `.shuffle()`, `.batch()`, `.prefetch()`, `.map()`
- **Save/Load**: `model.save()`, `keras.models.load_model()`, `tf.train.Checkpoint`

In [ ]:
# Quick reference — copy-paste snippets

# ── Imports ───────────────────────────────────────────────────────────────────
# import tensorflow as tf
# from tensorflow import keras
# from tensorflow.keras import layers

# ── Create ────────────────────────────────────────────────────────────────────
# tf.constant([1.0, 2.0, 3.0])
# tf.zeros([m, n])
# tf.random.normal([m, n])
# v = tf.Variable(initial)  →  v.assign(new)  /  v.assign_add(delta)

# ── Ops ───────────────────────────────────────────────────────────────────────
# a + b   a - b   a * b   a / b      # element-wise
# A @ B                               # matrix multiply
# tf.linalg.matmul(A, B)

# ── Shape ─────────────────────────────────────────────────────────────────────
# tf.reshape(t, [r, c])
# tf.expand_dims(t, axis)
# tf.squeeze(t, axis)
# tf.transpose(t, perm=[2, 0, 1])
# tf.concat([a, b], axis)
# tf.stack([a, b], axis)

# ── Autograd ──────────────────────────────────────────────────────────────────
# with tf.GradientTape() as tape:
#     y = f(x)
# grads = tape.gradient(y, [x, w, b])

# ── Neural network ────────────────────────────────────────────────────────────
# layers.Dense(units, activation)
# layers.Conv2D(filters, kernel_size)
# layers.BatchNormalization()
# model.compile(optimizer, loss, metrics)
# model.fit(train_ds, epochs, validation_data=val_ds)

# ── Custom training loop ──────────────────────────────────────────────────────
# with tf.GradientTape() as tape:
#     logits = model(x, training=True)
#     loss   = loss_fn(y, logits)
# grads = tape.gradient(loss, model.trainable_variables)
# opt.apply_gradients(zip(grads, model.trainable_variables))

# ── Save / Load ───────────────────────────────────────────────────────────────
# model.save("model.keras")
# keras.models.load_model("model.keras")

print("Quick reference loaded — uncomment and run any snippet above.")